# 03 · SMLP4Rec + destination prior + same-destination history (late fusion and hybrid)

Replaces `scripts/late_fusion_destination_prior.py` and `scripts/hybrid_samedest_comparison.py` (branch `exp/late-fusion-destination-prior`). No retraining: the model is notebook 01's best checkpoint.

```
fusion: score(k) = log p_SMLP4Rec(k | history) + w_p * log p_prior(k | searched destination)
hybrid: score(k) = log p_SMLP4Rec(k | history) + w_p * log p_prior(k | dest) + w_s * sameDest(k)
```

- **p_prior**: destination cluster counts smoothed to the destination's market, `(n_dest,k + m * p_market,k) / (n_dest + m)`, m = 5; unseen destination → global. Market = modal `hotel_market` of the destination in the prior data, never the event's own `hotel_market`. Counts use only bookings before the last train target (minus 128 s: RecBole stores time as float32), so no valid/test booking feeds its own prior.
- **sameDest(k)**: sum of 0.7^age over the user's past bookings of cluster k made at the query destination (age 0 = most recent).
- **Weights** chosen on VALID by Recall@5 (ties → smaller weights), per history bucket L = 1 | 2-4 | 5-9 | 10+ or one global value; test is read once.
- **Warm rows** = the RecBole 80/10/10 targets (L >= 1). **Cold rows** (each user's first booking, L = 0) are scored separately with the prior only, in the same time windows; "all events" = warm + cold.
- **Metrics**: Recall@K and NDCG@K (K = 5, 10, 20); paired bootstrap 95% intervals for Recall@5 and NDCG@5 differences on test.
- **Change from the scripts**: the scripts' secondary metrics (MRR@10, MAP@5) are dropped, and variant 4c is chosen by valid NDCG@5 instead of MAP@5.

Kernel: `smlp4rec`. Needs `data/interim/recbole/expedia_dest/expedia_dest.inter` (`scripts/expedia_to_recbole.py --with-destination`, same rows and order as the plain export) and the raw `train.csv` for the prior.

In [1]:
import functools
import json
import logging
import sys
import time
from pathlib import Path

import numpy as np
import torch  # before pandas (Windows c10.dll issue)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.compare import compare_nested
from nbp.eval.metrics import summarize, target_rank
from src.models.smlprec import SMLPREC

torch.autograd.set_detect_anomaly(False)
logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
RAW = ROOT / "data" / "raw" / "hospitality" / "expedia" / "train.csv"
INTER_DEST = WORK / "expedia_dest" / "expedia_dest.inter"
REF_DIR = ROOT / "reports" / "summary" / "week3"
RUN_DIR = sorted((ROOT / "experiments").glob("*_expedia_smlp4rec_week3-repro"))[-1]

N_CLUSTERS = 100
M_SMOOTH = 5
EPS = 1e-6
TIME_MARGIN = 128  # seconds; float32 timestamp resolution near 1.4e9
RECENCY = 0.7
W_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0]  # fusion sweep
W_P = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]  # hybrid grid
W_S = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0]
BUCKETS = [("L=1", 1, 1), ("L=2-4", 2, 4), ("L=5-9", 5, 9), ("L>=10", 10, 10**9)]
KS = (5, 10, 20)
print("run dir:", RUN_DIR)

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-02_expedia_smlp4rec_week3-repro


## 1. Data with the searched destination

Same split as notebook 01, plus `srch_destination_id` per booking, so every target row carries its query destination and every history position its booking's destination.

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_dest",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "srch_destination_id"]},
    },
)
init_seed(config["seed"], config["reproducibility"])
t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
tok2id = dataset.field2token_id[dataset.iid_field]
cluster_ids = np.array([tok2id[str(k)] for k in range(N_CLUSTERS)])  # model column of each raw cluster
n_items = dataset.item_num
ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
TIME = config["TIME_FIELD"]


def split_arrays(loader) -> dict:
    """History ids/length/destinations, target, query destination, past-cluster mask of one split."""
    d = loader.dataset.inter_feat
    seq, length = d[ITEM_SEQ].numpy(), d[ITEM_SEQ_LEN].numpy()
    y = d[config["ITEM_ID_FIELD"]].numpy()
    in_hist = np.zeros((len(y), n_items), bool)
    for j in range(seq.shape[1]):
        r = np.flatnonzero(j < length)
        in_hist[r, seq[r, j]] = True
    in_hist[:, 0] = False
    return {
        "seq": seq,
        "seq_t": d[ITEM_SEQ],
        "length": length,
        "length_t": d[ITEM_SEQ_LEN],
        "y": y,
        "old": in_hist[np.arange(len(y)), y],
        "q_dest": d["srch_destination_id"].numpy().astype(np.int64),
        "h_dest": d["srch_destination_id" + config["LIST_SUFFIX"]].numpy().astype(np.int64),
    }


arr = {"valid": split_arrays(valid_data), "test": split_arrays(test_data)}
masks = {sp: {b: (a["length"] >= lo) & (a["length"] <= hi) for b, lo, hi in BUCKETS} for sp, a in arr.items()}
print({sp: len(a["y"]) for sp, a in arr.items()}, f"{time.time() - t0:.0f}s")

{'valid': 218670, 'test': 218670} 61s


## 2. Destination prior (leak-free cut)

In [3]:
import pandas as pd  # after torch

t_train_max = float(train_data.dataset.inter_feat[TIME].max())
t_valid_max = float(valid_data.dataset.inter_feat[TIME].max())
cut = t_train_max - TIME_MARGIN


def build_prior_tables(cut_seconds: float) -> dict:
    """Cluster count tables from bookings with unix time < cut_seconds (one chunked pass over train.csv)."""
    cols = ["date_time", "is_booking", "srch_destination_id", "hotel_market", "hotel_cluster"]
    dtypes = {c: "int32" for c in cols if c != "date_time"}
    parts = []
    for ch in pd.read_csv(RAW, usecols=cols, chunksize=2_000_000, dtype=dtypes):
        ch = ch[ch.is_booking == 1]
        ts = (pd.to_datetime(ch["date_time"]) - pd.Timestamp("1970-01-01")) // pd.Timedelta(seconds=1)
        parts.append(ch.loc[ts.values < cut_seconds, ["srch_destination_id", "hotel_market", "hotel_cluster"]])
    b = pd.concat(parts, ignore_index=True)

    def table(key):
        t = b.groupby([key, "hotel_cluster"]).size().unstack(fill_value=0)
        return t.reindex(columns=range(N_CLUSTERS), fill_value=0)

    dest, mkt = table("srch_destination_id"), table("hotel_market")
    dm = b.groupby(["srch_destination_id", "hotel_market"]).size().reset_index(name="n")
    modal = dm.sort_values("n", ascending=False).drop_duplicates("srch_destination_id")
    glob = np.bincount(b["hotel_cluster"].values, minlength=N_CLUSTERS).astype(np.float64)
    return {
        "n_bookings": len(b),
        "dest": {int(d): r.astype(np.float64) for d, r in zip(dest.index, dest.values)},
        "market": {int(m): r.astype(np.float64) for m, r in zip(mkt.index, mkt.values)},
        "dest2mkt": dict(zip(modal["srch_destination_id"].astype(int), modal["hotel_market"].astype(int))),
        "global": glob / glob.sum(),
    }


def prior_vector(d: int) -> np.ndarray:
    """p(cluster | destination), shape (100,), raw cluster order."""
    row = tables["dest"].get(d)
    if row is None:
        return tables["global"]
    pm = tables["market"][tables["dest2mkt"][d]]
    pm = pm / pm.sum()
    return (row + M_SMOOTH * pm) / (row.sum() + M_SMOOTH)


def prior_logp(dests: np.ndarray) -> np.ndarray:
    """(n, n_items) log prior, columns = model item ids; padding column 0 stays 0 (masked at ranking)."""
    uniq, inv = np.unique(dests, return_inverse=True)
    tab = np.stack([np.log(prior_vector(int(d)) + EPS) for d in uniq]).astype(np.float32)
    out = np.zeros((len(dests), n_items), np.float32)
    out[:, cluster_ids] = tab[inv]
    return out


t1 = time.time()
tables = build_prior_tables(cut)
print(f"prior cut (unix s) {cut:.0f}; {tables['n_bookings']} bookings, {len(tables['dest'])} destinations; {time.time() - t1:.0f}s")

prior cut (unix s) 1412776576; 2463082 bookings, 34506 destinations; 34s


## 3. Score components per split

- `ml`: log-softmax of the model scores (padding column -inf).
- `pr`: log prior of the query destination.
- `sd`: sameDest matrix.

`rank_rows` sets the padding column to -inf on the array it is given (always a freshly built score matrix) and returns the rank of the true cluster.

In [4]:
model = SMLPREC(config, dataset)
model.load_state_dict(torch.load(RUN_DIR / "best.pth")["state_dict"])


@torch.no_grad()
def model_logp(a: dict, batch: int = 8192) -> np.ndarray:
    model.eval()
    out = []
    for i in range(0, len(a["y"]), batch):
        s = model.full_sort_predict(
            {model.ITEM_SEQ: a["seq_t"][i : i + batch], model.ITEM_SEQ_LEN: a["length_t"][i : i + batch]}
        )
        s[:, 0] = float("-inf")
        out.append(torch.log_softmax(s, dim=1).numpy())
    return np.concatenate(out)


def same_dest_matrix(a: dict) -> np.ndarray:
    """(n, n_items) sum of 0.7^age over past bookings of each cluster at the query destination."""
    s = np.zeros((len(a["y"]), n_items), np.float32)
    for j in range(a["seq"].shape[1]):
        idx = np.flatnonzero((j < a["length"]) & (a["h_dest"][:, j] == a["q_dest"]))
        s[idx, a["seq"][idx, j]] += RECENCY ** (a["length"][idx] - 1 - j)
    return s


def same_dest_rule(a: dict, prior: np.ndarray) -> np.ndarray:
    """No-learning rule: past clusters at the query destination first (recency 0.7^age), then the prior."""
    s = prior.copy()
    for j in range(a["seq"].shape[1]):
        idx = np.flatnonzero((j < a["length"]) & (a["h_dest"][:, j] == a["q_dest"]))
        bonus = 1000.0 * RECENCY ** (a["length"][idx] - 1 - j)
        cl = a["seq"][idx, j]
        s[idx, cl] = np.maximum(s[idx, cl], prior[idx, cl] + bonus)
    return s


def rank_rows(scores: np.ndarray, y: np.ndarray) -> np.ndarray:
    scores[:, 0] = -np.inf
    return target_rank(scores, y)


def fused(ml, pr, w) -> np.ndarray:
    """ml + w * pr; w scalar or per-row (n,)."""
    w = np.asarray(w, np.float32)
    return ml + (w[:, None] if w.ndim else w) * pr


ml, pr, sd = {}, {}, {}
for sp, a in arr.items():
    ml[sp], pr[sp], sd[sp] = model_logp(a), prior_logp(a["q_dest"]), same_dest_matrix(a)

ref01 = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
for sp in arr:
    got = summarize(rank_rows(fused(ml[sp], pr[sp], 0.0), arr[sp]["y"]))["recall@5"]
    print(f"sanity {sp}: plain Recall@5 {got} vs notebook 01 best epoch {ref01[sp]['recall@5']}")

sanity valid: plain Recall@5 0.3352 vs notebook 01 best epoch 0.3352


sanity test: plain Recall@5 0.3369 vs notebook 01 best epoch 0.3369


## 4. Late fusion: sweep w_p, choose on valid

In [5]:
sweep = {sp: {} for sp in arr}
for w in W_GRID:
    for sp, a in arr.items():
        rk = rank_rows(fused(ml[sp], pr[sp], w), a["y"])
        sweep[sp][str(w)] = {"all": summarize(rk), **{b: summarize(rk, m) for b, m in masks[sp].items()}}

chosen_w = {b: max(W_GRID, key=lambda w: (sweep["valid"][str(w)][b]["recall@5"], -w)) for b, _, _ in BUCKETS}
best_global_w = max(W_GRID, key=lambda w: (sweep["valid"][str(w)]["all"]["recall@5"], -w))
print("chosen w_p per bucket:", chosen_w, "| best global w_p:", best_global_w)


def per_row_w(sp: str) -> np.ndarray:
    w = np.zeros(len(arr[sp]["y"]), np.float32)
    for b, m in masks[sp].items():
        w[m] = chosen_w[b]
    return w


fusion_variants, fusion_ranks = {}, {}
for sp, a in arr.items():
    variants = {
        "plain_model (w=0)": fused(ml[sp], pr[sp], 0.0),
        "destination_prior_only": pr[sp] + 1e-6 * np.nan_to_num(ml[sp], neginf=0.0),
        "same_dest_history_then_prior": same_dest_rule(a, pr[sp]),
        f"fusion_global_w={best_global_w}": fused(ml[sp], pr[sp], best_global_w),
        "fusion_per_bucket_w": fused(ml[sp], pr[sp], per_row_w(sp)),
    }
    fusion_variants[sp], fusion_ranks[sp] = {}, {}
    for name, sc in variants.items():
        rk = rank_rows(sc, a["y"])
        fusion_ranks[sp][name] = rk
        fusion_variants[sp][name] = {
            "all": summarize(rk),
            "old_target_rows": summarize(rk, a["old"]),
            "new_target_rows": summarize(rk, ~a["old"]),
            **{b: summarize(rk, m) for b, m in masks[sp].items()},
        }
pd.DataFrame({k: v["all"] for k, v in fusion_variants["test"].items()}).T

chosen w_p per bucket: {'L=1': 2.0, 'L=2-4': 1.5, 'L=5-9': 1.5, 'L>=10': 1.5} | best global w_p: 1.5


,n,recall@5,recall@10,recall@20,ndcg@5,ndcg@10,ndcg@20
plain_model (w=0),218670.0,0.3369,0.4645,0.6309,0.2439,0.2850,0.3269
destination_prior_only,218670.0,0.5298,0.7212,0.8845,0.3631,0.4251,0.4666
same_dest_history_then_prior,218670.0,0.5888,0.7548,0.8971,0.4377,0.4915,0.5276
fusion_global_w=1.5,218670.0,0.5919,0.7657,0.9086,0.4227,0.4791,0.5154
fusion_per_bucket_w,218670.0,0.5912,0.7656,0.9084,0.4221,0.4786,0.5150


## 5. Cold users (L = 0): prior only

First booking of every user, valid window = [last train target, last valid target), test window = from the last valid target on.

In [6]:
inter = pd.read_csv(INTER_DEST, sep="\t").sort_values(["user_id:token", "timestamp:float"], kind="stable")
first = inter.drop_duplicates("user_id:token", keep="first")
windows = {
    "valid": (first["timestamp:float"] >= t_train_max) & (first["timestamp:float"] < t_valid_max),
    "test": first["timestamp:float"] >= t_valid_max,
}
glob_row = np.full(n_items, -np.inf, np.float32)
glob_row[cluster_ids] = np.log(tables["global"] + EPS)

cold_rank, cold = {}, {}
for sp, mask in windows.items():
    f = first[mask]
    y = np.array([tok2id[str(int(c))] for c in f["item_id:token"]])
    dests = f["srch_destination_id:float"].values.astype(np.int64)
    cold_rank[sp] = {
        "prior": rank_rows(prior_logp(dests), y),
        "global": rank_rows(np.tile(glob_row, (len(y), 1)), y),
    }
    n_dest = np.array([tables["dest"][d].sum() if d in tables["dest"] else 0 for d in dests])
    support = {"n_dest=0": n_dest == 0, "n_dest 1-19": (n_dest >= 1) & (n_dest < 20), "n_dest>=20": n_dest >= 20}
    cold[sp] = {
        "n_cold_rows": len(y),
        "prior_only": summarize(cold_rank[sp]["prior"]),
        "global_popularity": summarize(cold_rank[sp]["global"]),
        "prior_only_by_destination_support": {
            k: {**summarize(cold_rank[sp]["prior"], m), "share": round(float(m.mean()), 4)} for k, m in support.items()
        },
    }
    print(sp, "cold rows", len(y), "| prior-only", cold[sp]["prior_only"], "\n   global", cold[sp]["global_popularity"])

combined = {}
for sp in arr:
    warm, n_warm = fusion_ranks[sp], len(arr[sp]["y"])
    cp, cg = cold_rank[sp]["prior"], cold_rank[sp]["global"]
    combined[sp] = {
        "n_warm_rows": n_warm,
        "n_cold_rows": len(cp),
        "cold_share_of_events": round(len(cp) / (len(cp) + n_warm), 4),
        "hybrid: fusion on warm + prior-only on cold": summarize(np.concatenate([warm[f"fusion_global_w={best_global_w}"], cp])),
        "prior-only for everyone": summarize(np.concatenate([warm["destination_prior_only"], cp])),
        "week-2 heuristic (same-dest history, prior when cold)": summarize(np.concatenate([warm["same_dest_history_then_prior"], cp])),
        "plain SMLP4Rec on warm + global popularity on cold": summarize(np.concatenate([warm["plain_model (w=0)"], cg])),
    }
pd.DataFrame({k: v for k, v in combined["test"].items() if isinstance(v, dict)}).T

valid cold rows 50803 | prior-only {'n': 50803, 'recall@5': 0.529, 'recall@10': 0.7158, 'recall@20': 0.8849, 'ndcg@5': 0.3614, 'ndcg@10': 0.4219, 'ndcg@20': 0.4649} 
   global {'n': 50803, 'recall@5': 0.1251, 'recall@10': 0.2101, 'recall@20': 0.3651, 'ndcg@5': 0.08, 'ndcg@10': 0.1073, 'ndcg@20': 0.1461}


test cold rows 49443 | prior-only {'n': 49443, 'recall@5': 0.5123, 'recall@10': 0.7017, 'recall@20': 0.8751, 'ndcg@5': 0.3506, 'ndcg@10': 0.4119, 'ndcg@20': 0.456} 
   global {'n': 49443, 'recall@5': 0.1317, 'recall@10': 0.2269, 'recall@20': 0.3634, 'ndcg@5': 0.0813, 'ndcg@10': 0.1118, 'ndcg@20': 0.1458}


,n,recall@5,recall@10,recall@20,ndcg@5,ndcg@10,ndcg@20
hybrid: fusion on warm + prior-only on cold,268113.0,0.5772,0.7539,0.9024,0.4094,0.4667,0.5045
prior-only for everyone,268113.0,0.5266,0.7176,0.8828,0.3608,0.4226,0.4646
"week-2 heuristic (same-dest history, prior when cold)",268113.0,0.5747,0.7450,0.8931,0.4217,0.4768,0.5144
plain SMLP4Rec on warm + global popularity on cold,268113.0,0.2990,0.4206,0.5815,0.2139,0.2530,0.2935


## 6. Hybrid with sameDest: grid (w_p, w_s), choose on valid

In [7]:
GRID = [(wp, ws) for wp in W_P for ws in W_S]
t2 = time.time()
rk_grid = {sp: {c: rank_rows(ml[sp] + c[0] * pr[sp] + c[1] * sd[sp], arr[sp]["y"]) for c in GRID} for sp in arr}
print(f"grid {len(GRID)} pairs x 2 splits: {time.time() - t2:.0f}s")


def best_combo(mask: np.ndarray, metric: str, grid: list) -> tuple:
    """Grid pair with the best valid Recall@5 or NDCG@5 in the mask; ties -> smaller weights."""

    def key(c):
        rk = rk_grid["valid"][c][mask]
        v = (rk <= 5).mean() if metric == "recall" else ((rk <= 5) / np.log2(rk + 1)).mean()
        return (-round(float(v), 6), c[0] + c[1])

    return min(grid, key=key)


buckets = [b for b, _, _ in BUCKETS]
fusion_grid = [c for c in GRID if c[1] == 0.0]
all_valid = np.ones(len(arr["valid"]["y"]), bool)
chosen = {
    "fusion_global": {b: best_combo(all_valid, "recall", fusion_grid) for b in buckets},
    "hybrid_global": {b: best_combo(all_valid, "recall", GRID) for b in buckets},
    "hybrid_per_bucket": {b: best_combo(masks["valid"][b], "recall", GRID) for b in buckets},
    "hybrid_per_bucket_by_ndcg": {b: best_combo(masks["valid"][b], "ndcg", GRID) for b in buckets},
}
for k, v in chosen.items():
    print(f"{k:28s} {v}")


def per_bucket_rank(sp: str, ch: dict) -> np.ndarray:
    out = np.zeros(len(arr[sp]["y"]), np.int64)
    for b, m in masks[sp].items():
        out[m] = rk_grid[sp][ch[b]][m]
    return out


warm_rank = {}
for sp, a in arr.items():
    warm_rank[sp] = {
        "plain": rk_grid[sp][(0.0, 0.0)],
        "prior_only": fusion_ranks[sp]["destination_prior_only"],
        "same_dest_rule (no model)": fusion_ranks[sp]["same_dest_history_then_prior"],
        **{name: per_bucket_rank(sp, ch) for name, ch in chosen.items()},
    }

LABELS = {
    "plain": "1. plain SMLP4Rec",
    "prior_only": "2. regional prior only",
    "fusion_global": "3. SMLP4Rec + prior (w_p)",
    "hybrid_per_bucket": "4. hybrid: + sameDest (per bucket, by Recall@5)",
    "hybrid_global": "4b. hybrid, one global (w_p, w_s)",
    "hybrid_per_bucket_by_ndcg": "4c. hybrid, per bucket, chosen by NDCG@5",
    "same_dest_rule (no model)": "ref. same-destination rule (no model)",
}


def cold_for(key: str, sp: str) -> np.ndarray:
    """Cold rows: global popularity for the plain model (no cold handling), prior otherwise."""
    return cold_rank[sp]["global" if key == "plain" else "prior"]


hyb_warm, hyb_all = {}, {}
for sp, a in arr.items():
    hyb_warm[sp], hyb_all[sp] = {}, {}
    for key, label in LABELS.items():
        rk = warm_rank[sp][key]
        hyb_warm[sp][label] = {
            "all": summarize(rk),
            "old_target_rows": summarize(rk, a["old"]),
            "new_target_rows": summarize(rk, ~a["old"]),
            **{b: summarize(rk, m) for b, m in masks[sp].items()},
        }
        hyb_all[sp][label] = summarize(np.concatenate([rk, cold_for(key, sp)]))
    hyb_all[sp]["cold rows only: prior"] = summarize(cold_rank[sp]["prior"])

print("TEST, users with history (warm)")
display(pd.DataFrame({k: v["all"] for k, v in hyb_warm["test"].items()}).T)
print("TEST, all events (warm + cold)")
display(pd.DataFrame(hyb_all["test"]).T)

grid 42 pairs x 2 splits: 17s


fusion_global                {'L=1': (1.5, 0.0), 'L=2-4': (1.5, 0.0), 'L=5-9': (1.5, 0.0), 'L>=10': (1.5, 0.0)}
hybrid_global                {'L=1': (2.0, 8.0), 'L=2-4': (2.0, 8.0), 'L=5-9': (2.0, 8.0), 'L>=10': (2.0, 8.0)}
hybrid_per_bucket            {'L=1': (2.0, 8.0), 'L=2-4': (1.5, 8.0), 'L=5-9': (2.0, 8.0), 'L>=10': (1.5, 8.0)}
hybrid_per_bucket_by_ndcg    {'L=1': (2.0, 8.0), 'L=2-4': (2.0, 8.0), 'L=5-9': (2.0, 8.0), 'L>=10': (2.0, 8.0)}


TEST, users with history (warm)


,n,recall@5,recall@10,recall@20,ndcg@5,ndcg@10,ndcg@20
1. plain SMLP4Rec,218670.0,0.3369,0.4645,0.6309,0.2439,0.2850,0.3269
2. regional prior only,218670.0,0.5298,0.7212,0.8845,0.3631,0.4251,0.4666
3. SMLP4Rec + prior (w_p),218670.0,0.5919,0.7657,0.9086,0.4227,0.4791,0.5154
"4. hybrid: + sameDest (per bucket, by Recall@5)",218670.0,0.6074,0.7727,0.9104,0.4498,0.5035,0.5385
"4b. hybrid, one global (w_p, w_s)",218670.0,0.6071,0.7722,0.9097,0.4501,0.5036,0.5386
"4c. hybrid, per bucket, chosen by NDCG@5",218670.0,0.6071,0.7722,0.9097,0.4501,0.5036,0.5386
ref. same-destination rule (no model),218670.0,0.5888,0.7548,0.8971,0.4377,0.4915,0.5276


TEST, all events (warm + cold)


,n,recall@5,recall@10,recall@20,ndcg@5,ndcg@10,ndcg@20
1. plain SMLP4Rec,268113.0,0.2990,0.4206,0.5815,0.2139,0.2530,0.2935
2. regional prior only,268113.0,0.5266,0.7176,0.8828,0.3608,0.4226,0.4646
3. SMLP4Rec + prior (w_p),268113.0,0.5772,0.7539,0.9024,0.4094,0.4667,0.5045
"4. hybrid: + sameDest (per bucket, by Recall@5)",268113.0,0.5899,0.7596,0.9039,0.4316,0.4866,0.5233
"4b. hybrid, one global (w_p, w_s)",268113.0,0.5896,0.7592,0.9034,0.4318,0.4867,0.5234
"4c. hybrid, per bucket, chosen by NDCG@5",268113.0,0.5896,0.7592,0.9034,0.4318,0.4867,0.5234
ref. same-destination rule (no model),268113.0,0.5747,0.7450,0.8931,0.4217,0.4768,0.5144
cold rows only: prior,49443.0,0.5123,0.7017,0.8751,0.3506,0.4119,0.4560


## 7. Paired bootstrap on test (1000 resamples, seed 0)

In [8]:
COMPARISONS = [
    ("hybrid_per_bucket", "same_dest_rule (no model)"),
    ("hybrid_per_bucket", "fusion_global"),
    ("hybrid_per_bucket", "prior_only"),
    ("fusion_global", "prior_only"),
    ("fusion_global", "same_dest_rule (no model)"),
    ("hybrid_per_bucket", "plain"),
]
boot = {}
for ka, kb in COMPARISONS:
    ra, rb = warm_rank["test"][ka], warm_rank["test"][kb]
    boot[f"{ka} minus {kb}"] = {
        "warm": paired_bootstrap(ra, rb, k=5),
        "all_events": paired_bootstrap(
            np.concatenate([ra, cold_for(ka, "test")]), np.concatenate([rb, cold_for(kb, "test")]), k=5
        ),
    }
pd.DataFrame(
    {
        (k, scope): {f"{m} diff": v[scope][m]["diff"] for m in ("recall@5", "ndcg@5")}
        | {f"{m} ci95": v[scope][m]["ci95"] for m in ("recall@5", "ndcg@5")}
        for k, v in boot.items()
        for scope in ("warm", "all_events")
    }
).T

recall@5 diff  \
hybrid_per_bucket minus same_dest_rule (no model) warm              0.0186   
                                                  all_events        0.0151   
hybrid_per_bucket minus fusion_global             warm              0.0155   
                                                  all_events        0.0126   
hybrid_per_bucket minus prior_only                warm              0.0776   
                                                  all_events        0.0633   
fusion_global minus prior_only                    warm              0.0621   
                                                  all_events        0.0506   
fusion_global minus same_dest_rule (no model)     warm              0.0031   
                                                  all_events        0.0025   
hybrid_per_bucket minus plain                     warm              0.2705   
                                                  all_events        0.2908   

                                                             ndcg@5 diff  \
hybrid_per_bucket minus same_dest_rule (no model) warm            0.0121   
                                                  all_events      0.0099   
hybrid_per_bucket minus fusion_global             warm            0.0272   
                                                  all_events      0.0221   
hybrid_per_bucket minus prior_only                warm            0.0867   
                                                  all_events      0.0707   
fusion_global minus prior_only                    warm            0.0596   
                                                  all_events      0.0486   
fusion_global minus same_dest_rule (no model)     warm            -0.015   
                                                  all_events     -0.0123   
hybrid_per_bucket minus plain                     warm            0.2059   
                                                  all_events      0.2176   

                                                                 recall@5 ci95  \
hybrid_per_bucket minus same_dest_rule (no model) warm          [0.0171, 0.02]   
                                                  all_events   [0.014, 0.0162]   
hybrid_per_bucket minus fusion_global             warm        [0.0148, 0.0163]   
                                                  all_events  [0.0121, 0.0133]   
hybrid_per_bucket minus prior_only                warm        [0.0759, 0.0793]   
                                                  all_events  [0.0619, 0.0646]   
fusion_global minus prior_only                    warm        [0.0603, 0.0638]   
                                                  all_events   [0.0492, 0.052]   
fusion_global minus same_dest_rule (no model)     warm        [0.0014, 0.0047]   
                                                  all_events  [0.0012, 0.0037]   
hybrid_per_bucket minus plain                     warm        [0.2682, 0.2727]   
                                                  all_events  [0.2887, 0.2928]   

                                                                     ndcg@5 ci95  
hybrid_per_bucket minus same_dest_rule (no model) warm          [0.0112, 0.0131]  
                                                  all_events    [0.0091, 0.0106]  
hybrid_per_bucket minus fusion_global             warm          [0.0265, 0.0279]  
                                                  all_events    [0.0216, 0.0227]  
hybrid_per_bucket minus prior_only                warm          [0.0853, 0.0882]  
                                                  all_events    [0.0697, 0.0718]  
fusion_global minus prior_only                    warm          [0.0583, 0.0608]  
                                                  all_events    [0.0475, 0.0495]  
fusion_global minus same_dest_rule (no model)     warm        [-0.0163, -0.0138]  
                                                  all_events  [-0.0133, -0.0113]  
hybrid_per_bucket minus plain                     warm          [0.2044, 0.20

## 8. Save and compare with the script runs

Results are reshaped to the scripts' JSON layout; every common numeric value is compared (Recall@K, NDCG@10, chosen weights, row counts, bootstrap Recall@5 intervals). The scripts' MRR / MAP values and variant 4c have no counterpart here by design.

In [9]:
fusion_result = {
    "prior": {"smoothing_m": M_SMOOTH, "eps": EPS, "cutoff_unix_seconds": cut, "n_bookings": tables["n_bookings"], "n_destinations": len(tables["dest"])},
    "w_grid": W_GRID,
    "chosen_w_per_bucket_on_valid_recall@5": chosen_w,
    "best_global_w_on_valid": best_global_w,
    "rows": {sp: len(arr[sp]["y"]) for sp in arr},
    "share_of_rows_per_bucket": {sp: {b: round(float(m.mean()), 4) for b, m in masks[sp].items()} for sp in arr},
    "variants": fusion_variants,
    "cold_users_prior_only": cold,
    "combined_warm_plus_cold": combined,
    "sweep": sweep,
}
hybrid_result = {
    "grids": {"w_p": W_P, "w_s": W_S},
    "chosen_weights_on_valid": {k: {b: list(v) for b, v in ch.items()} for k, ch in chosen.items()},
    "rows": {sp: len(arr[sp]["y"]) for sp in arr},
    "cold_rows": {sp: len(cold_rank[sp]["prior"]) for sp in arr},
    "warm": hyb_warm,
    "all_events": hyb_all,
    "bootstrap_test_95ci": boot,
}
(RUN_DIR / "late_fusion.json").write_text(json.dumps(fusion_result, indent=2), encoding="utf-8")
(RUN_DIR / "hybrid_samedest.json").write_text(json.dumps(hybrid_result, indent=2), encoding="utf-8")

rows = []
for ref_file, res in (("smlprec_expedia_late_fusion.json", fusion_result), ("smlprec_expedia_hybrid_samedest.json", hybrid_result)):
    ref = json.loads((REF_DIR / ref_file).read_text(encoding="utf-8"))
    rows += [(ref_file, *r) for r in compare_nested(ref, res)]
cmp = pd.DataFrame(rows, columns=["reference", "path", "script", "notebook", "diff"])
pd.set_option("display.max_rows", 300)
print(f"compared {len(cmp)} values; identical: {(cmp['diff'].abs() < 1e-9).sum()}; max |diff|: {cmp['diff'].abs().max():.6f}")
print(cmp.groupby("reference")["diff"].agg(lambda d: f"{(d.abs() < 1e-9).sum()}/{len(d)} identical, max |diff| {d.abs().max():.6f}"))
cmp[cmp["diff"].abs() >= 1e-9] if (cmp["diff"].abs() >= 1e-9).any() else cmp.head(20)

compared 1550 values; identical: 1550; max |diff|: 0.000000
reference
smlprec_expedia_hybrid_samedest.json    567/567 identical, max |diff| 0.000000
smlprec_expedia_late_fusion.json        983/983 identical, max |diff| 0.000000
Name: diff, dtype: object


,reference,path,script,notebook,diff
0,smlprec_expedia_late_fusion.json,prior.smoothing_m,5.000000e+00,5.000000e+00,0.0
1,smlprec_expedia_late_fusion.json,prior.eps,1.000000e-06,1.000000e-06,0.0
2,smlprec_expedia_late_fusion.json,prior.n_bookings,2.463082e+06,2.463082e+06,0.0
3,smlprec_expedia_late_fusion.json,prior.n_destinations,3.450600e+04,3.450600e+04,0.0
4,smlprec_expedia_late_fusion.json,w_grid.0,0.000000e+00,0.000000e+00,0.0
5,smlprec_expedia_late_fusion.json,w_grid.1,1.000000e-01,1.000000e-01,0.0
6,smlprec_expedia_late_fusion.json,w_grid.2,2.500000e-01,2.500000e-01,0.0
7,smlprec_expedia_late_fusion.json,w_grid.3,5.000000e-01,5.000000e-01,0.0
8,smlprec_expedia_late_fusion.json,w_grid.4,7.500000e-01,7.500000e-01,0.0
9,smlprec_expedia_late_fusion.json,w_grid.5,1.000000e+00,1.000000e+00,0.0
